### Imports

In [ ]:
# ! pip install mlflow datasets langchain langchain-google-genai

In [19]:
import mlflow
import os
import json
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.rate_limiters import InMemoryRateLimiter
from mlflow.genai.optimize import GepaPromptOptimizer

from mlflow.genai import scorer

load_dotenv()

os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:5000"
os.environ["MLFLOW_EXPERIMENT_NAME"] = "prompt_optimization"

### Load the dataset and save 50 samples

#### Uncomment these cells when running for the first time

In [2]:
# Load AG News dataset from Hugging Face as pandas dataframe
from datasets import load_dataset
dataset = load_dataset("ag_news", split="train")
df = dataset.to_pandas()

df["label"] = df["label"].map({0: "World", 1: "Sports", 2: "Business", 3: "Science"})

df = df.sample(frac=1).reset_index(drop=True)

df.head()

d:\youtube\TheAIGuy\NLP\mlflow_examples\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,text,label
0,Infocus: Detecting Worms and Abnormal Activiti...,Science
1,"Global Markets: Shares Rise, Bonds Fall LONDO...",Business
2,Iliadis Takes Greece #39;s Second Gold with Ju...,Sports
3,Israel Pulls Back Forces in Northern Gaza JAB...,World
4,Apple opens Canadian iTunes store Apple missed...,Science


In [3]:
NUM_SAMPLES = 20
train_data = []
for i in range(NUM_SAMPLES):
    article = df.iloc[i]["text"]
    expected = df.iloc[i]["label"]
    eval_dict = {
        "inputs": {"article": article},
        "expectations": {"expected_response": expected},
    }
    train_data.append(eval_dict)

train_data[0]

{'inputs': {'article': 'Infocus: Detecting Worms and Abnormal Activities with NetFlow, Part 1 This paper discusses the use of NetFlow, a traffic profile monitoring technology available on many routers, for use in the early detection of worms, spammers, and other abnormal network activity in large enterprise networks and service providers.'},
 'expectations': {'expected_response': 'Science'}}

In [ ]:
# # Save train_data list to a jsonl file
# import json
# with open("train_data.jsonl", "w") as f:
#     json.dump(train_data, f)

#### Load the training data from saved path

In [ ]:
# with open("train_data.jsonl", "r") as f:
#     train_data = json.load(f)

# train_data[0]

### Initialize the llm

In [6]:
rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.1,  # <-- Super slow! We can only make a request once every 10 seconds!!
    check_every_n_seconds=0.1,  # Wake up every 100 ms to check whether allowed to make a request,
    max_bucket_size=10,  # Controls the maximum burst size.
)

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    rate_limiter=rate_limiter,
)

### Register an initial prompt

In [17]:
# Create a detailed prompt for classification
prompt = """
You are a helpful assistant that can classify news articles into one of the following categories:
- World
- Sports
- Business
- Science
Article: {article}
"""

initial_prompt = mlflow.genai.register_prompt(
    name="news_classifier",
    template=prompt,
)

2025/10/24 22:10:25 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: news_classifier, version 2


### Create a predict function and test with a sample

In [7]:
def predict_fn(article) -> str:
    prompt_template = mlflow.genai.load_prompt("news_classifier", version=1).template
    prompt = prompt_template.format(article=article)
    response = llm.invoke(prompt)
    return response.content

In [8]:
predict_fn(train_data[0]["inputs"]["article"])

'Science'

### Create an baseline

In [9]:
@scorer
def exact_match(outputs, expectations):
    expectations = expectations["expected_response"]
    return outputs == expectations

with mlflow.start_run(run_name="optimize-prompt-baseline"):
    results = mlflow.genai.evaluate(
        data=train_data,
        scorers=[exact_match],
        predict_fn=predict_fn,
    )

2025/10/24 22:02:40 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset.
Evaluating:  75%|███████▌  | 15/20 [Elapsed: 03:06, Remaining: 01:02] Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit.
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 10
Please retry in 4.547315131s. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "locati

### Optimize the prompt

In [22]:
result = mlflow.genai.optimize_prompts(
    predict_fn=predict_fn,
    train_data=train_data,
    prompt_uris=[initial_prompt.uri],
    optimizer=GepaPromptOptimizer(reflection_model="openai:/gpt-5"),
    scorers=[exact_match],
)

2025/10/24 22:13:33 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset.
d:\youtube\TheAIGuy\NLP\mlflow_examples\.venv\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: Failed to determine whether UCVolumeDatasetSource can resolve source information for 'prompt_optimization_train_data'. Exception: 
  return _dataset_source_registry.resolve(
d:\youtube\TheAIGuy\NLP\mlflow_examples\.venv\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more in

Iteration 0: Base program full valset score: 0.9
Iteration 1: Selected program 0 score: 0.9
Iteration 1: All subsample scores perfect. Skipping.
Iteration 1: Reflective mutation did not propose a new candidate
Iteration 2: Selected program 0 score: 0.9
Iteration 2: All subsample scores perfect. Skipping.
Iteration 2: Reflective mutation did not propose a new candidate
Iteration 3: Selected program 0 score: 0.9
Iteration 3: All subsample scores perfect. Skipping.
Iteration 3: Reflective mutation did not propose a new candidate
Iteration 4: Selected program 0 score: 0.9
Iteration 4: Proposed new text for news_classifier: You are a classification assistant. Given a news article’s text, output exactly one category label from this fixed set (case-sensitive):
- World
- Sports
- Business
- Science

Output rules:
- Output only the single label (no explanations, punctuation, or extra text).
- Choose the label that best matches the article’s primary focus.
- If the article touches multiple areas

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised InternalServerError: 500 An internal error has occurred. Please retry or report in https://developers.generativeai.google/guide/troubleshooting.
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised InternalServerError: 500 An internal error has occurred. Please retry or report in https://developers.generativeai.google/guide/troubleshooting.
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised InternalServerError: 500 An internal error has occurred. Please retry or report in https://developers.generativeai.google/guide/troubleshooting.
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised InternalServerError: 500 An internal error has occurred. Please retry or report in https://developers.gen

Iteration 4: Full valset score for new program: 0.8
Iteration 4: Full train_val score for new program: 0.8
Iteration 4: Individual valset scores for new program: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0]
Iteration 4: New valset pareto front scores: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
Iteration 4: Full valset pareto front score: 1.0
Iteration 4: Updated valset pareto front programs: [{0, 1}, {0, 1}, {0, 1}, {0, 1}, {1}, {0, 1}, {0, 1}, {1}, {0, 1}, {0, 1}, {0, 1}, {0, 1}, {0}, {0, 1}, {0}, {0}, {0}, {0, 1}, {0, 1}, {0, 1}]
Iteration 4: Best valset aggregate score so far: 0.9
Iteration 4: Best program as per aggregate score on train_val: 0
Iteration 4: Best program as per aggregate score on valset: 0
Iteration 4: Best score on valset: 0.9
Iteration 4: Best score on train_val: 0.9
Iteration 4: Linear pareto front program index: 0
Iteration 4: New program candidate in

2025/10/24 22:29:20 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: news_classifier, version 3


Iteration 12: New program is on the linear pareto front
Iteration 12: Full valset score for new program: 1.0
Iteration 12: Full train_val score for new program: 1.0
Iteration 12: Individual valset scores for new program: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
Iteration 12: New valset pareto front scores: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
Iteration 12: Full valset pareto front score: 1.0
Iteration 12: Updated valset pareto front programs: [{0, 1, 2}, {0, 1, 2}, {0, 1, 2}, {0, 1, 2}, {1, 2}, {0, 1, 2}, {0, 1, 2}, {1, 2}, {0, 1, 2}, {0, 1, 2}, {0, 1, 2}, {0, 1, 2}, {0, 2}, {0, 1, 2}, {0, 2}, {0, 2}, {0, 2}, {0, 1, 2}, {0, 1, 2}, {0, 1, 2}]
Iteration 12: Best valset aggregate score so far: 1.0
Iteration 12: Best program as per aggregate score on train_val: 2
Iteration 12: Best program as per aggregate score on valset: 2
Iteration 12: Best score on valset: 1.0
Ite

### Run the evaluation with Optimized prompt

In [25]:
def predict_fn(article) -> str:
    prompt_template = mlflow.genai.load_prompt("news_classifier", version=3).template
    prompt = prompt_template.format(article=article)
    response = llm.invoke(prompt)
    return response.content

In [26]:
sample_result = predict_fn(train_data[0]["inputs"]["article"])
print(sample_result)

Science


In [28]:
@scorer
def exact_match(outputs, expectations):
    expectations = expectations["expected_response"]
    # outputs = json.loads(outputs.replace("```json\n", "").replace("\n```", ""))[
    #     "expected_response"
    # ]
    print(f"Outputs: {outputs}, Expectations: {expectations}")
    return outputs == expectations


with mlflow.start_run(run_name="optimized-prompt-eval"):
    results = mlflow.genai.evaluate(
        data=train_data,
        scorers=[exact_match],
        predict_fn=predict_fn,
    )

2025/10/24 22:36:51 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset.
Evaluating:   5%|▌         | 1/20 [Elapsed: 00:04, Remaining: 01:19] 

Outputs: Science, Expectations: Science


Evaluating:  10%|█         | 2/20 [Elapsed: 00:05, Remaining: 00:51] 

Outputs: Business, Expectations: Business


Evaluating:  15%|█▌        | 3/20 [Elapsed: 00:08, Remaining: 00:46] 

...

Outputs: World, Expectations: World


Evaluating:  95%|█████████▌| 19/20 [Elapsed: 02:19, Remaining: 00:07] 

Outputs: Sports, Expectations: Sports


Evaluating: 100%|██████████| 20/20 [Elapsed: 02:32, Remaining: 00:00] 

Outputs: World, Expectations: World
